In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import seaborn as sns


In [ ]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

fichier_data = project_root / "data" / "raw" / "dereev_data_rea.xlsx"
colonne_notes = "cn_notes"
colonne_reev = "reev_classe"

df = pd.read_excel(fichier_data)

colonnes_requises = [colonne_notes, colonne_reev]
colonnes_manquantes = [col for col in colonnes_requises if col not in df.columns]
if colonnes_manquantes:
    raise ValueError(
        f"Colonnes manquantes : {colonnes_manquantes}. "
        f"Colonnes disponibles : {list(df.columns)}"
    )

nb_lignes_initial = len(df)
df = df.dropna(subset=colonnes_requises).copy()
df[colonne_notes] = df[colonne_notes].astype(str).str.strip()
df = df[df[colonne_notes] != ""].copy()
df[colonne_reev] = df[colonne_reev].astype(int)

labels_invalides = sorted(set(df[colonne_reev]) - {0, 1})
if labels_invalides:
    raise ValueError(f"Labels invalides dans {colonne_reev} : {labels_invalides}")

print(f"Fichier charge : {fichier_data}")
print(f"{nb_lignes_initial} lignes trouvees dans le fichier brut.")
print(f"{len(df)} notes exploitables apres suppression des valeurs manquantes.")
print(df[colonne_reev].value_counts().sort_index())


In [ ]:
### PIE CHART REPARTITION DES CLASSES

sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 6))

repartition = df[colonne_reev].value_counts().reindex([0, 1], fill_value=0)
plt.pie(
    repartition,
    labels=["Pas de reevaluation : 0", "Reevaluation : 1"],
    autopct="%1.1f%%",
    startangle=90,
    explode=(0, 0.05),
    shadow=True,
    colors=["#4C72B0", "#C44E52"],
)

plt.title("Repartition des reevaluations", fontweight="bold")
plt.show()


In [ ]:
### WORDCLOUD

texte_0 = " ".join(df.loc[df[colonne_reev] == 0, colonne_notes])
texte_1 = " ".join(df.loc[df[colonne_reev] == 1, colonne_notes])

wc_0 = WordCloud(width=600, height=400, background_color="white", colormap="Blues", max_words=100).generate(texte_0)
wc_1 = WordCloud(width=600, height=400, background_color="white", colormap="Reds", max_words=100).generate(texte_1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(wc_0, interpolation="bilinear")
axes[0].axis("off")
axes[0].set_title("Notes sans reevaluation (0)", fontsize=16, color="#4C72B0", pad=15)

axes[1].imshow(wc_1, interpolation="bilinear")
axes[1].axis("off")
axes[1].set_title("Notes avec reevaluation (1)", fontsize=16, color="#C44E52", pad=15)

plt.tight_layout()
plt.show()


In [ ]:
df["longueur_mots"] = df[colonne_notes].apply(lambda x: len(x.split()))

print(f"Moyenne      : {df['longueur_mots'].mean():.2f} mots")
print(f"Mediane      : {df['longueur_mots'].median():.0f} mots")
print(f"Maximum      : {df['longueur_mots'].max()} mots")
print(f"Minimum      : {df['longueur_mots'].min()} mots")
print(f"90e centile  : {df['longueur_mots'].quantile(0.90):.0f} mots")

sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 6))

sns.histplot(data=df, x="longueur_mots", bins=30, kde=True, color="#4C72B0", edgecolor="black")

moyenne = df["longueur_mots"].mean()
mediane = df["longueur_mots"].median()
plt.axvline(moyenne, color="red", linestyle="--", linewidth=2, label=f"Moyenne ({moyenne:.2f})")
plt.axvline(mediane, color="green", linestyle="-", linewidth=2, label=f"Mediane ({mediane:.0f})")

plt.title("Distribution de la longueur des notes (en mots)", fontsize=16, pad=15, fontweight="bold")
plt.xlabel("Nombre de mots par note", fontsize=12)
plt.ylabel("Nombre de notes", fontsize=12)
plt.legend(fontsize=11)

plt.tight_layout()
plt.show()
